# Qualitative browser — base vs FT output pairs

Side-by-side raw generations for the same eval problem across two run tags (role of vlm-alignment's `chat-ui.ipynb`). Reads `runs/<tag>/<model>-<arm>/results.jsonl` + `eval_v1` references.

In [ ]:
# CONFIG
MODEL, ARM = "8b", "story"
TAG_A, TAG_B = "base-v1", "ft-v1"
BUCKET = "unparseable"   # show problems where TAG_A landed in this bucket
N = 5

In [ ]:
import json, pathlib
RUNS, EVAL = pathlib.Path("..") / "runs", pathlib.Path("..") / "eval_v1"
refs = {r["problem_id"]: r for t in ("normal", "hard", "extra_hard", "order5")
        for r in map(json.loads, (EVAL / f"eval_{t}.jsonl").read_text().splitlines())}
def load(tag):
    p = RUNS / tag / f"{MODEL}-{ARM}" / "results.jsonl"
    return {r["pair_id"]: r for r in map(json.loads, p.read_text().splitlines())} if p.exists() else {}
a, b = load(TAG_A), load(TAG_B)
shown = 0
for pid, ra in a.items():
    if ra["bucket"] != BUCKET or pid not in b:
        continue
    rb = b[pid]
    print("=" * 72)
    print(f"{pid}  {TAG_A}:{ra['bucket']}  ->  {TAG_B}:{rb['bucket']}")
    print("REF:", refs[pid]["reference_rg"].replace("\n", "  |  "))
    print(f"[{TAG_A}]\n{ra['response'][:400]}")
    print(f"[{TAG_B}]\n{rb['response'][:400]}")
    shown += 1
    if shown >= N:
        break
print(f"({shown} pairs shown)")